In [22]:
# ========== 导入：三人对话练习要用的库 ==========

# 标准库 os：读 GEMINI_API_KEY 等环境变量（Environment Variables）
import os
# 标准库 random：随机选谁先开口、打乱其余发言顺序
import random
# Google GenAI 官方包：原生调用 Gemma / Gemini
from google import genai
# OpenAI 客户端：这里用来对接本地 Ollama 的兼容接口
from openai import OpenAI
# types：构造 Content / Part / GenerateContentConfig 等结构体
from google.genai import types
# load_dotenv：从 .env 加载密钥，避免写进代码
from dotenv import load_dotenv
# 网页抓取辅助（本练习主路径未必用到；原 import 保留）
from scraper import fetch_website_contents
# IPython 显示：需要时用 Markdown 渲染
from IPython.display import Markdown, display


In [23]:
# ========== 环境变量：加载并粗检 GEMINI_API_KEY ==========

# 在名为 .env 的文件中加载环境变量；override=True 表示覆盖已有同名变量
load_dotenv(override=True)
# 读取 Google / Gemini 侧的 API Key
api_key = os.getenv('GEMINI_API_KEY')

# 检查钥匙：以下 print 文案用于排错，英文保持原样（可能被脚本/习惯依赖）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Gemma 4 API 调用

下面用 **Google GenAI 原生客户端** 扮演朋友 **C**（害羞、短回复）。
模型 id、system 指令字符串保持英文原样，不要改。


In [24]:
# ========== 初始化 Google GenAI 客户端 ==========

# 初始化本机 Google 客户端
# 它会自动获取 GEMINI_API_KEY 环境变量（也可显式传 api_key=）
client = genai.Client()


In [3]:
# ========== call_gemma：用 Gemma 扮演朋友 C ==========

def call_gemma(message):
    # 1. 用户留言：包装成 GenAI 的 Content（role=user + Part 文本）
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    ]

    # 2. 放置系统规则：性格=shy，硬性字数限制（英文指令不翻译）
    config = types.GenerateContentConfig(
        system_instruction="You are 'C' part of 3 young friends in a conversation, your personality is shy. Answer in under 60 characters. Do not explain. Be brief.",
    )
    # 3. 使用原生客户端调用模型（model id 保持原样）
    response = client.models.generate_content(
        model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
        contents=message,
        config=config
    )

    # 返回纯文本，供主循环打印/存档
    return response.text


# Ollama 调用

用 **OpenAI 兼容接口** 访问本机 Ollama：朋友 **A** 用 `llama3.2`，朋友 **B** 用 `deepseek-r1:1.5b`。
请先保证 `ollama serve` 已启动，并 pull 好对应模型。


In [4]:
# ========== 连接本地 Ollama（OpenAI 兼容 base_url） ==========

# Ollama 默认地址；/v1 表示兼容 OpenAI Chat Completions
ollama_url = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；base_url 指向本机
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [5]:
# ========== call_ollama：按人物选模型并完成一轮回复 ==========

# person：'A' 或 'B'；mensaje：上一句对话内容（变量名保持原样）
def call_ollama(person, mensaje):
    # A → llama3.2；B → deepseek-r1:1.5b（model id 不改）
    if person == "A":
        MODEL = "llama3.2"
    elif person == "B":
        MODEL = "deepseek-r1:1.5b"
    # 组装 system/user messages（见 other_messages），再请求本地模型
    ol1_response = ollama.chat.completions.create(
        model = MODEL,
        messages = other_messages(person, mensaje)
    )
    # 取出助手文本内容
    return ol1_response.choices[0].message.content


# 三人对话 —— 暂无完善的异常处理

模拟朋友 **A / B / C** 轮流聊天：
- 第 1 轮随机选人开场，其余两人顺序打乱
- 之后每一轮：上一轮的「第二发言者」变成新开场人

想加第四人 **D** 时，可扩展 `persons` 与 `get_dialog` 里的台词表（原代码已留注释位）。


In [6]:
# ========== 台词表 + 存档：为每人准备可复用的开场句 ==========

def get_dialog(person, dialog_num, turn):
    """模拟一个人的对话。"""
    # 每人一组英文台词（影响模型上下文的可运行字符串，不翻译）
    messages = {
        "A": [
            "Hey everyone, what are we doing this weekend?",
            "I was thinking we could go hiking.",
            "The weather forecast looks great for Saturday.",
            "Should we pack lunch or eat out?",
            "Let's meet at the trailhead at 9am then!",
        ],
        "B": [
            "Welcome! I'm so glad you're here, guys.",
            "Hiking sounds awesome, I know a great trail.",
            "Perfect, I'll bring my camera then.",
            "Let's pack lunch, it's cheaper and more fun.",
            "Sounds like a plan, I'll bring extra water!",
        ],
        "C": [
            "Can you make sandwiches?",
            "Oh I love hiking, which trail are you thinking?",
            "Great, I'll charge my phone for photos.",
            "I can make sandwiches for everyone if you want.",
            "Can't wait, this is going to be so fun!",
        ],
        "D": [
            "Hi there",
            "Count me in! ",
            "Oh I love hiking",
            "Great",
            "Can't wait!",
        ],
    }
    # 使用 dialog_num 与 turn 选择稍微不同的行（取模循环）
    idx = (dialog_num + turn) % len(messages[person])
    # 拼成 "A: ...." 这种前缀形式
    f_dialog = f"{person}: {messages[person][idx]}"
    # 返回 [人物字母, 去掉前缀后的正文] —— 与后面解包 first_speaker, f_dialog 对齐
    return [f_dialog[:1], f_dialog[3:]]

def store_line(person, line):
    """将对话行存储在适当的列表中。"""
    # line 形如 {"A": "..."}，追加到对应全局列表
    if person == "A":
        listA.append(line)
    elif person == "B":
        listB.append(line)
    elif person == "C":
        listC.append(line)
    # elif 人==“D”：
    # listC.append(行)


In [12]:
# ========== 开场人 + 消息构造（Gemma / Ollama） ==========

def first_dialog():
    """随机获取某人的第一条消息"""
    # 从 persons 里随机挑一个先说话
    first_speaker = random.choice(persons)  # Random starter for dialog 1
    # dialog_num=0, turn=0：取该人台词表里对应下标
    starting_dialog = get_dialog(first_speaker, 0, 0)
    return starting_dialog


def gemma_messages(message):
    # 把纯文本包成 GenAI Content 列表（辅助函数；主路径未必调用）
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    ]
    return messages


def other_messages(person, message):
    """为 Ollama 路径构造 system + user messages。"""
    # 按人物写入不同性格（英文 personality 字符串不改）
    if person == "A":
        personality = "You are 'A', part of 3 young friends in a conversation, your personality is inpacient."
    elif person == "B":
        personality = "You are 'B', part of 3 young friends in a conversation, your personality is bored."
    # system：性格 + 60 字符硬限制；user：上一句对话
    messages = [
    {"role": "system",
    "content": f"{personality} You have a hard budget of 60 characters total. Output ONLY the clean answer string. Do not think. Do not explain."},
    {"role": "user", "content": message}
    ]
    return messages


In [20]:
# ========== 主循环：多轮三人对话编排 ==========

def get_responses(person, message):
    """获取人员的回复（辅助封装；主循环里更多是直接 call_*）。"""
    if person == "A" or person == "B":
        # 注意：原代码这里传的是字面量 f"f_dialog"，逻辑保持不动
        result = call_ollama(person, f"f_dialog")
    elif person == "C":
        result = call_gemma(f_dialog)
    return result

def run_llm_conversation(num_dialogs=5):
    """运行“num_dialogs”轮对话。

    每轮对话有 3 轮（每人一轮）：
      - 对话 1：随机启动，然后其他两个按随机顺序排列。
      - 对话 N (N > 1)：前一个对话的第一响应者（第二个发言者）
        对话框成为新的启动器，然后其余两个按随机顺序排列。"""
    # 解包：先说话的人 + 其开场文本
    first_speaker, f_dialog = first_dialog()  # Random starter for dialog 1

    # 打印分隔标题（英文展示字符串保持原样）
    print("=" * 50)
    print("        THREE-PERSON CONVERSATION")
    print("=" * 50)

    for dialog_num in range(num_dialogs):
        # 为此对话构建有序发言人列表：开场人固定，其余 shuffle
        others = [p for p in persons if p != first_speaker]
        random.shuffle(others)
        order = [first_speaker] + others  # first speaker is fixed; others shuffled

        print(f"\n--- Dialog {dialog_num + 1} ---")
        print(f"  Turn order: {' → '.join(order)}")

        for turn, speaker in enumerate(order):
            # 第 1 轮第 1 个 turn：直接用预设开场句，不调模型
            if dialog_num == 0 and turn == 0:
                line = {speaker: f_dialog}
            else:
                # A/B 走 Ollama；C 走 Gemma
                if speaker == "A" or speaker == "B":
                    if speaker == "A":
                      # 若 A 已有历史，取上一句作为上下文
                      if len(listA) != 0:
                          f_dialog = listA[-1]['A']
                    elif speaker == "B":
                      if len(listB) != 0:
                          f_dialog = listB[-1]['B']
                    result = call_ollama(speaker, f_dialog)
                    # 去掉换行，方便单行打印
                    line = {speaker: result.replace('\n','')}
                elif speaker == "C":
                    if len(listC) != 0:
                        f_dialog = listC[-1]['C']
                    result = call_gemma(f_dialog)
                    line = {speaker: result}
            # 存档并打印本 turn
            store_line(speaker, line)
            print(f"  {line}")

        # 规则 3：下一个对话的起始者是该对话的第一响应者（第二发言者）
        first_speaker = order[1]

    print("\n" + "=" * 50)


In [ ]:
# ========== 启动：全局历史列表 + 跑 5 轮 ==========

# 每个人的对话的存储列表（供下一 turn 取「上一句」）
listA = []
listB = []
listC = []

# 参与者名单；想加 D 时取消注释并补全 store_line 等分支
persons = ["A", "B", "C"] # , "D"]

# 跑 5 轮三人对话（需要本机 Ollama + 有效 GEMINI_API_KEY）
run_llm_conversation(5)
